# 📦 Capstone Project Part 5.1 – Week 12: Multi-Agent Weather Information Assistant

---

## 🎯 Title:
**Designing a Function-Calling Agent to Retrieve and Format Live Weather Data Using OpenAI and WeatherAPI**

---

## 📝 Purpose:
This project demonstrates the use of function-calling with OpenAI’s GPT to build a simple multi-agent system. The agent is capable of interpreting natural language queries, calling a real-time weather API, and formatting the results for human-readable output. This exercise reflects how Large Language Models (LLMs) can coordinate tasks, integrate external data sources, and act as autonomous problem-solvers in practical domains like logistics, travel, and customer service.

---

## ✅ Learning Outcomes:

- Apply function-calling to enable real-world task execution using LLMs
- Integrate OpenAI GPT models with external APIs such as WeatherAPI
- Handle user queries using dynamic parameter extraction and execution
- Format API responses into natural language summaries
- Implement basic error handling for API failures and input issues


---

## 🧩 Task 1: Weather API Integration

In this task, we build the foundational weather retrieval function by integrating the [WeatherAPI](https://www.weatherapi.com/). This function will later be called by our OpenAI agent to retrieve real-time weather data.

### 🛠️ Steps:

1. Import the `requests` module and the `WEATHER_API_KEY` from the `config.py` file.
2. Define a function `get_weather(location)` that makes an HTTP request to the WeatherAPI and returns the current weather as JSON.
3. Implement basic error handling for common API issues (e.g., connectivity or malformed requests).
4. Test the weather function by retrieving the current weather for a known location (e.g., London).

### 📌 Expected Output:

A successful printout of:
- The location name
- The current temperature in °C
- The weather condition description (e.g., "Partly cloudy")

This verifies that our API credentials are working and the data is correctly formatted.


In [1]:
# Keys are read from the environment - copy .env.example to .env and fill
# it in. Never hardcode credentials in a notebook.
import os

from dotenv import load_dotenv

load_dotenv()

# weather_agent.py - Task 1: Weather API Integration

import requests

# API Key (set it directly here for Colab use)
WEATHER_API_KEY = os.environ["WEATHER_API_KEY"]

def get_weather(location):
    try:
        url = "http://api.weatherapi.com/v1/current.json"
        params = {
            "key": WEATHER_API_KEY,
            "q": location,
            "aqi": "no"
        }
        response = requests.get(url, params=params)
        response.raise_for_status()  # Raise exception for bad status codes
        return response.json()
    except requests.exceptions.RequestException as e:
        raise Exception(f"Weather API error: {str(e)}")

# ✅ Test weather API integration
if __name__ == "__main__":
    try:
        weather_data = get_weather("London")
        print(f"Weather in {weather_data['location']['name']}:")
        print(f"Temperature: {weather_data['current']['temp_c']}°C")
        print(f"Condition: {weather_data['current']['condition']['text']}")
    except Exception as e:
        print(f"Error: {e}")



Weather in London:
Temperature: 18.3°C
Condition: Partly cloudy


# 💬 Part 2: Adding OpenAI Function Definition

import openai

# Set your OpenAI key directly for Colab use
OPENAI_API_KEY = "sk-..."  # Replace with your OpenAI key

openai.api_key = OPENAI_API_KEY

# Define the function that OpenAI can call
functions = [
    {
        "name": "get_weather",
        "description": "Get current weather for a location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City name or location"
                }
            },
            "required": ["location"]
        }
    }
]


In [2]:
# Keys are read from the environment - copy .env.example to .env and fill
# it in. Never hardcode credentials in a notebook.
import os

from dotenv import load_dotenv

load_dotenv()

# 💬 Part 2: Adding OpenAI Function Definition

import openai

# Set your OpenAI key directly for Colab use
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]  # Replace with your OpenAI key

openai.api_key = OPENAI_API_KEY

# Define the function that OpenAI can call
functions = [
    {
        "name": "get_weather",
        "description": "Get current weather for a location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City name or location"
                }
            },
            "required": ["location"]
        }
    }
]


In [3]:
# ✅ Test OpenAI function definition

try:
    response = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[
            {"role": "user", "content": "What's the weather like in Tokyo?"}
        ],
        functions=functions,
        function_call="auto"
    )

    print("OpenAI Response:")
    print(response.choices[0].message.function_call)

except Exception as e:
    print(f"Error: {e}")


OpenAI Response:
{
  "name": "get_weather",
  "arguments": "{\n  \"location\": \"Tokyo\"\n}"
}


In [8]:
!pip install openai==0.28 --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 1.9 MB/s eta 0:00:00


## 🌦️ Part 3: Implementing the Weather Agent

In this section, we complete the agent workflow by enabling OpenAI to dynamically call the weather function and return real-time results. The agent parses the user's query, selects the appropriate function, fetches weather data via API, and delivers a natural language response.

This simulates a retrieval-augmented generation (RAG) setup, where function execution augments the language model’s capabilities.


In [4]:
# ✅ Part 3: Implementing the Weather Agent

def weather_agent(user_query):
    try:
        # Get function call from OpenAI
        response = openai.ChatCompletion.create(
            model="gpt-4",
            messages=[{"role": "user", "content": user_query}],
            functions=functions,
            function_call="auto"
        )

        # Extract function call
        function_call = response.choices[0].message.function_call

        # Execute function if it matches
        if function_call.name == "get_weather":
            import json
            arguments = json.loads(function_call.arguments)
            weather_data = get_weather(arguments["location"])

            # Format response
            return (
                f"Current weather in {weather_data['location']['name']}: "
                f"{weather_data['current']['temp_c']}°C, "
                f"{weather_data['current']['condition']['text']}"
            )
        else:
            return "No matching function call found."

    except Exception as e:
        return f"Error: {str(e)}"

# ✅ Final Test – Run sample queries
test_queries = [
    "What's the weather like in Singapore?",
    "Is it raining in London right now?",
    "Tell me the temperature in New York"
]

for query in test_queries:
    print(f"\nQuery: {query}")
    print(f"Response: {weather_agent(query)}")



Query: What's the weather like in Singapore?
Response: Current weather in Singapore: 32.3°C, Partly cloudy

Query: Is it raining in London right now?
Response: Current weather in London: 18.3°C, Partly cloudy

Query: Tell me the temperature in New York
Response: Current weather in New York: 28.9°C, Thundery outbreaks in nearby


In [5]:
def weather_agent(query):
    try:
        response = openai.ChatCompletion.create(
            model="gpt-4",
            messages=[
                {"role": "user", "content": query}
            ],
            functions=functions,
            function_call="auto"
        )
        func_call = response.choices[0].message.function_call
        if func_call.name == "get_weather":
            location = eval(func_call.arguments)["location"]
            weather_data = get_weather(location)

            # Emoji mapping based on condition text
            condition_text = weather_data['current']['condition']['text'].lower()
            emoji_map = {
                "sunny": "☀️", "clear": "🌙",
                "cloudy": "☁️", "partly cloudy": "🌤️",
                "rain": "🌧️", "showers": "🌦️",
                "thunder": "⛈️", "snow": "❄️",
                "fog": "🌫️"
            }
            matched_emoji = next((emoji for key, emoji in emoji_map.items() if key in condition_text), "🌈")

            return (
                f"📍 Weather in {weather_data['location']['name']}:\n"
                f"{matched_emoji} Condition: {weather_data['current']['condition']['text']}\n"
                f"🌡 Temperature: {weather_data['current']['temp_c']}°C / {weather_data['current']['temp_f']}°F\n"
                f"🥵 Feels Like: {weather_data['current']['feelslike_c']}°C\n"
                f"💧 Humidity: {weather_data['current']['humidity']}%\n"
                f"💨 Wind: {weather_data['current']['wind_kph']} kph {weather_data['current']['wind_dir']}\n"
                f"🕒 Updated: {weather_data['current']['last_updated']}"
            )
        else:
            return "❌ No matching function call found."

    except Exception as e:
        return f"❗ Error: {str(e)}"


### ✅ Bonus Features: Weather Agent Enhancement

This enhanced version of the `weather_agent()` function includes the following improvements for better user experience and assignment completeness:

- **Weather emojis** based on condition keywords for visual clarity (☀️🌧️🌫️).
- **Both Celsius and Fahrenheit** temperature output for accessibility.
- **Feels like temperature**, **humidity**, and **wind speed/direction** for more context.
- **Time of last weather update** to ensure recency of information.
- **Error handling** for unmatched function calls and exception reporting.

The final query block demonstrates these improvements by simulating natural user questions and printing detailed, user-friendly responses.


In [6]:
# 🧪 Final Test with Enhanced Weather Agent
test_queries = [
    "What's the weather like in Singapore?",
    "Tell me the temperature in New York",
    "Is it raining in London right now?"
]

for query in test_queries:
    print(f"\n🔎 Query: {query}")
    print(weather_agent(query))



🔎 Query: What's the weather like in Singapore?
📍 Weather in Singapore:
☁️ Condition: Partly cloudy
🌡 Temperature: 32.1°C / 89.8°F
🥵 Feels Like: 40.1°C
💧 Humidity: 59%
💨 Wind: 20.5 kph S
🕒 Updated: 2025-06-26 13:45

🔎 Query: Tell me the temperature in New York
📍 Weather in New York:
⛈️ Condition: Thundery outbreaks in nearby
🌡 Temperature: 28.9°C / 84.0°F
🥵 Feels Like: 34.7°C
💧 Humidity: 61%
💨 Wind: 10.8 kph NNE
🕒 Updated: 2025-06-26 01:45

🔎 Query: Is it raining in London right now?
📍 Weather in London:
☁️ Condition: Partly cloudy
🌡 Temperature: 18.3°C / 64.9°F
🥵 Feels Like: 18.3°C
💧 Humidity: 83%
💨 Wind: 15.5 kph WSW
🕒 Updated: 2025-06-26 06:45


## 📄 Submission Summary: Weather Agent with RAG Integration

This notebook demonstrates a fully working **Weather Agent** that uses:
- **WeatherAPI** for real-time weather data
- **OpenAI GPT-4 with function calling** for natural language interaction

---

### ✅ What Was Implemented

#### 🔧 Core Features
- WeatherAPI integration with dynamic location queries
- OpenAI function calling setup (`get_weather`)
- Natural language questions (e.g. "Is it raining in London?") trigger the correct API call

#### 🎁 Bonus Enhancements
- Emoji support based on weather conditions (e.g., ☀️, 🌧️, 🌫️)
- Dual temperature display: Celsius and Fahrenheit
- Added contextual info: Feels like temp, humidity, wind speed/direction
- Timestamp of weather reading for clarity

---

### 🛠️ Error Handling Overview

| Area                 | Error Handled                                  |
|----------------------|------------------------------------------------|
| WeatherAPI           | Connection errors, bad responses               |
| OpenAI API           | Invalid API keys, model errors, fallback logic |
| User Queries         | Unmatched function calls return clear messages |
| API Responses        | Fallback for missing condition emoji mappings  |

---

### 📸 Output Evidence
- Screenshot included showing successful responses to 3 user queries:
  - “What’s the weather like in Singapore?”
  - “Tell me the temperature in New York.”
  - “Is it raining in London right now?”

---

### ✅ Ready for Submission

- ✅ All requirements complete
- ✅ Bonus improvements included
- ✅ Functionality tested and confirmed
